# Bài tập 4 - Mạng Neural Đơn giản với PyTorch
Trong bài thực hành này, chúng ta sẽ xây dựng mạng neural đơn giản sử dụng **PyTorch**.
   <center> <img  src="./images/C2_W1_CoffeeRoasting.png" width="400" />   <center/>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
import torch
import torch.nn as nn
from collections import OrderedDict
from torch.utils.data import TensorDataset, DataLoader
from lab_coffee_torch_utils import load_coffee_data, plt_roast, plt_prob, plt_layer, plt_network, plt_output_unit
import logging

## Bộ dữ liệu (DataSet)

In [ ]:
X,Y = load_coffee_data();
print(X.shape, Y.shape)

Hãy vẽ biểu đồ dữ liệu rang cà phê bên dưới. Hai đặc trưng là Nhiệt độ theo độ C và Thời lượng theo phút. Trang [Coffee Roasting at Home](https://www.merchantsofgreencoffee.com/how-to-roast-green-coffee-in-your-oven/) gợi ý rằng thời lượng tốt nhất nên duy trì trong khoảng 12 đến 15 phút trong khi nhiệt độ nên nằm trong khoảng 175 đến 260 độ C. Tất nhiên, khi nhiệt độ tăng lên thì thời lượng nên giảm đi. 

In [ ]:
plt_roast(X,Y)

### Chuẩn hóa Dữ liệu (Normalize Data)
Việc khớp các trọng số với dữ liệu (lan truyền ngược - back-propagation) sẽ diễn ra nhanh hơn nếu dữ liệu được chuẩn hóa. 

Trong PyTorch, chúng ta có thể thực hiện chuẩn hóa Z-score trực tiếp bằng cách tính giá trị trung bình (mean) và độ lệch chuẩn (std) của tập dữ liệu huấn luyện, hoặc đóng gói thành một module `nn.Module` có phương thức `adapt()` tương tự Keras:
- Tính `mean` và `std` của tập dữ liệu `X`.
- Lưu lại các giá trị này để chuẩn hóa bất kỳ dữ liệu mới nào trong tương lai.
- Chuẩn hóa dữ liệu theo công thức: $z = \frac{x - \mu}{\sigma}$.

In [ ]:
print(f"Temperature Max, Min pre normalization: {np.max(X[:,0]):0.2f}, {np.min(X[:,0]):0.2f}")
print(f"Duration    Max, Min pre normalization: {np.max(X[:,1]):0.2f}, {np.min(X[:,1]):0.2f}")

# Xây dựng lớp chuẩn hóa theo phong cách PyTorch
class Normalization(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('mean', torch.zeros(2))
        self.register_buffer('std', torch.ones(2))
        
    def adapt(self, x):
        """Học mean và variance/std từ tập dữ liệu"""
        if isinstance(x, np.ndarray):
            x = torch.from_numpy(x).float()
        self.mean = x.mean(dim=0)
        self.std = x.std(dim=0, unbiased=False)
        
    def forward(self, x):
        """Chuẩn hóa dữ liệu đầu vào"""
        if isinstance(x, np.ndarray):
            x_t = torch.from_numpy(x).float()
            return ((x_t - self.mean) / self.std).numpy()
        return (x - self.mean) / self.std

norm_l = Normalization()
norm_l.adapt(X)  # học mean và std
Xn = norm_l(X)
print(f"Temperature Max, Min post normalization: {np.max(Xn[:,0]):0.2f}, {np.min(Xn[:,0]):0.2f}")
print(f"Duration    Max, Min post normalization: {np.max(Xn[:,1]):0.2f}, {np.min(Xn[:,1]):0.2f}")

Sao chép/nhân bản (Tile/copy) dữ liệu của chúng ta để tăng kích thước tập huấn luyện và giảm số lượng epoch cần huấn luyện.

In [ ]:
Xt = np.tile(Xn,(1000,1))
Yt= np.tile(Y,(1000,1))   
print(Xt.shape, Yt.shape)

## Mô hình PyTorch

### Mô hình
   <center> <img  src="./images/C2_W1_RoastingNetwork.PNG" width="200" />   <center/>  
Hãy cùng xây dựng mạng "Coffee Roasting Network". Mạng gồm có hai lớp với hàm kích hoạt sigmoid như minh họa dưới đây.


Trong PyTorch, chúng ta sử dụng `nn.Sequential` để xếp chồng các lớp. Ta có thể sử dụng `OrderedDict` để đặt tên cho từng layer (`layer1`, `layer2`), giúp dễ dàng truy xuất trọng số tương tự như trong Keras:

In [6]:
torch.manual_seed(1234)  # thiết lập seed để đảm bảo kết quả tái lập nhất quán

model = nn.Sequential(OrderedDict([
    ('layer1', nn.Linear(in_features=2, out_features=3)),
    ('sigmoid1', nn.Sigmoid()),
    ('layer2', nn.Linear(in_features=3, out_features=1)),
    ('sigmoid2', nn.Sigmoid())
]))

>**Lưu ý 1:** Trong PyTorch, `nn.Linear(in_features, out_features)` yêu cầu chúng ta chỉ định rõ số chiều đầu vào `in_features=2` và số đơn vị đầu ra `out_features=3`. Điều này giúp PyTorch khởi tạo ngay lập tức kích thước của các ma trận trọng số và bias.  
>**Lưu ý 2:** Giống như trong TensorFlow, việc đưa hàm kích hoạt sigmoid trực tiếp vào lớp cuối cùng không được xem là phương pháp tối ưu nhất (best practice). Thay vào đó, trong PyTorch ta thường dùng `nn.BCEWithLogitsLoss()` kết hợp lớp tuyến tính cuối cùng (không có sigmoid) để tối ưu độ ổn định về mặt số học. Tuy nhiên, để bám sát bài giảng và dễ trực quan hóa xác suất, ở đây chúng ta sẽ dùng lớp `nn.Sigmoid()` rõ ràng và hàm mất mát `nn.BCELoss()`.

In kiến trúc mạng:

In [ ]:
print(model)

Số lượng tham số tương ứng với số phần tử trong các mảng trọng số và bias như minh họa dưới đây:

In [ ]:
L1_num_params = 2 * 3 + 3   # W1 parameters (2x3) + b1 parameters (3)
L2_num_params = 3 * 1 + 1   # W2 parameters (3x1) + b2 parameters (1)
print("L1 params = ", L1_num_params, ", L2 params = ", L2_num_params)

# Đếm tổng số tham số trong mô hình PyTorch
total_params = sum(p.numel() for p in model.parameters())
print("Total params = ", total_params)

Hãy cùng kiểm tra các trọng số và bias mà PyTorch đã khởi tạo.  
> **Lưu ý quan trọng về PyTorch vs TensorFlow:**

> Trong PyTorch, trọng số của lớp `nn.Linear(in_features, out_features)` được lưu trữ với kích thước là `(out_features, in_features)` (vì PyTorch thực hiện phép tính $y = x W^T + b$). Trong khi đó, TensorFlow/Keras lưu trữ dưới dạng `(in_features, out_features)` ($y = x W + b$).  
> Do đó, để ma trận trọng số có kích thước `(2, 3)` và `(3, 1)` quen thuộc như trong bài giảng, chúng ta chuyển vị (`.T`) tensor trọng số khi trích xuất ra mảng NumPy:
- Ở lớp đầu tiên với 3 đơn vị (units), chúng ta kỳ vọng W có kích thước là (2,3) và $b$ có 3 phần tử.
- Ở lớp thứ hai với 1 đơn vị (unit), chúng ta kỳ vọng W có kích thước là (3,1) và $b$ có 1 phần tử.

In [ ]:
# Trích xuất trọng số và bias từ các lớp của PyTorch
W1 = model.layer1.weight.detach().numpy().T
b1 = model.layer1.bias.detach().numpy()
W2 = model.layer2.weight.detach().numpy().T
b2 = model.layer2.bias.detach().numpy()

print(f"W1{W1.shape}:\n", W1, f"\nb1{b1.shape}:", b1)
print(f"W2{W2.shape}:\n", W2, f"\nb2{b2.shape}:", b2)

Các bước huấn luyện trong PyTorch:
- Định nghĩa hàm mất mát (loss function): sử dụng `nn.BCELoss()` (Binary Cross Entropy Loss).
- Định nghĩa thuật toán tối ưu (optimizer): sử dụng `torch.optim.Adam` với tốc độ học `lr=0.01`.
- Đóng gói dữ liệu vào `TensorDataset` và `DataLoader` với `batch_size=32` (tương tự như mặc định của TensorFlow).
- Chạy vòng lặp huấn luyện (training loop) thực hiện thuật toán hạ gradient (gradient descent).

In [ ]:
# Chuyển đổi dữ liệu sang PyTorch Tensor
X_train = torch.tensor(Xt, dtype=torch.float32)
y_train = torch.tensor(Yt, dtype=torch.float32)

# Hàm mất mát và bộ tối ưu hóa
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# DataLoader với batch_size = 32
dataset = TensorDataset(X_train, y_train)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Vòng lặp huấn luyện qua 10 epochs
epochs = 10
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_X, batch_y in dataloader:
        optimizer.zero_grad()               # Xóa gradient cũ
        outputs = model(batch_X)            # Lan truyền xuôi (Forward pass)
        loss = criterion(outputs, batch_y)  # Tính độ mất mát
        loss.backward()                     # Lan truyền ngược (Backward pass)
        optimizer.step()                    # Cập nhật trọng số
        epoch_loss += loss.item() * batch_X.size(0)
    
    total_loss = epoch_loss / len(dataset)
    print(f"Epoch {epoch+1}/{epochs} - loss: {total_loss:.4f}")

#### Epochs và batches
Trong đoạn mã trên, số lượng `epochs` được đặt là 10. Điều này chỉ định rằng toàn bộ tập dữ liệu huấn luyện sẽ được duyệt qua 10 lần.
Tập dữ liệu mở rộng có 200.000 mẫu, với `batch_size=32` trong `DataLoader`, mỗi epoch sẽ thực thi $200.000 / 32 = 6.250$ batches. Quá trình hạ gradient được cập nhật sau mỗi batch.

#### Các trọng số đã được cập nhật
Sau khi huấn luyện (fitting), các trọng số đã được cập nhật: 

In [ ]:
W1 = model.layer1.weight.detach().numpy().T
b1 = model.layer1.bias.detach().numpy()
W2 = model.layer2.weight.detach().numpy().T
b2 = model.layer2.bias.detach().numpy()
print("W1:\n", W1, "\nb1:", b1)
print("W2:\n", W2, "\nb2:", b2)

Bạn có thể thấy rằng các giá trị đã khác so với những gì bạn đã in ra trước khi gọi huấn luyện. Với những trọng số này, mô hình sẽ có thể phân biệt được mẻ cà phê nào là rang đạt chuẩn hay không đạt chuẩn.

Để phục vụ cho phần thảo luận tiếp theo, thay vì sử dụng ngay các trọng số bạn vừa nhận được, trước tiên bạn sẽ thiết lập một số trọng số mà chúng tôi đã lưu từ một lần huấn luyện trước đó. Điều này nhằm giúp notebook này hoạt động ổn định và nhất quán. Các lần huấn luyện khác nhau có thể tạo ra các kết quả hơi khác nhau một chút và phần thảo luận sau đây áp dụng khi mô hình có các trọng số mà bạn sẽ tải dưới đây. 

Sau này, bạn có thể tự do chạy lại notebook với ô bên dưới được chú thích (comment out) để xem có sự khác biệt nào không. Nếu bạn đạt được độ mất mát thấp sau quá trình huấn luyện ở trên (ví dụ: 0.002), thì khả năng rất cao là bạn sẽ nhận được cùng kết quả.

In [12]:
# After finishing the lab later, you can re-run all 
# cells except this one to see if your trained model
# gets the same results.

# Set weights from a previous run. 
W1 = np.array([
    [-8.94,  0.29, 12.89],
    [-0.17, -7.34, 10.79]] )
b1 = np.array([-9.87, -9.28,  1.01])
W2 = np.array([
    [-31.38],
    [-27.86],
    [-32.79]])
b2 = np.array([15.54])

# Replace the weights from your trained model with the values above.
# Lưu ý: chuyển vị W (.T) vì PyTorch Linear lưu trữ dạng (out_features, in_features)
with torch.no_grad():
    model.layer1.weight.copy_(torch.tensor(W1.T, dtype=torch.float32))
    model.layer1.bias.copy_(torch.tensor(b1, dtype=torch.float32))
    model.layer2.weight.copy_(torch.tensor(W2.T, dtype=torch.float32))
    model.layer2.bias.copy_(torch.tensor(b2, dtype=torch.float32))

In [ ]:
# Check if the weights are successfully replaced
W1 = model.layer1.weight.detach().numpy().T
b1 = model.layer1.bias.detach().numpy()
W2 = model.layer2.weight.detach().numpy().T
b2 = model.layer2.bias.detach().numpy()
print("W1:\n", W1, "\nb1:", b1)
print("W2:\n", W2, "\nb2:", b2)

### Dự đoán (Predictions)
<img align="left" src="./images/C2_W1_RoastingDecision.PNG"     style=" width:380px; padding: 10px 20px; " >

Khi đã có một mô hình được huấn luyện, bạn có thể sử dụng nó để đưa ra dự đoán. Hãy nhớ rằng đầu ra của mô hình chúng ta là một xác suất. Trong trường hợp này, đó là xác suất mẻ rang đạt chuẩn (good roast). Để đưa ra quyết định, ta phải so sánh xác suất với một ngưỡng (threshold). Ở đây, chúng ta sẽ sử dụng ngưỡng 0.5

Hãy bắt đầu bằng cách tạo dữ liệu đầu vào. Mô hình kỳ vọng nhận vào một hoặc nhiều mẫu thử, trong đó các mẫu nằm trên các hàng của ma trận. Trong trường hợp này, chúng ta có hai đặc trưng nên ma trận sẽ có kích thước (m,2) với m là số lượng mẫu.
Hãy nhớ rằng, chúng ta đã chuẩn hóa các đặc trưng đầu vào nên chúng ta cũng phải chuẩn hóa dữ liệu kiểm tra của mình.   
Để đưa ra dự đoán trong PyTorch, ta đặt mô hình ở chế độ đánh giá (`model.eval()`) và tính toán đầu ra mà không cần tính gradient (`torch.no_grad()`).

In [ ]:
X_test = np.array([
    [200,13.9],  # positive example
    [200,17]])   # negative example
X_testn = norm_l(X_test)

model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_testn, dtype=torch.float32)
    predictions = model(X_test_tensor).numpy()

print("predictions = \n", predictions)

Để chuyển đổi các xác suất thành quyết định, chúng ta áp dụng một ngưỡng:

In [ ]:
yhat = np.zeros_like(predictions)
for i in range(len(predictions)):
    if predictions[i] >= 0.5:
        yhat[i] = 1
    else:
        yhat[i] = 0
print(f"decisions = \n{yhat}")

Điều này có thể được thực hiện một cách ngắn gọn hơn:

In [ ]:
yhat = (predictions >= 0.5).astype(int)
print(f"decisions = \n{yhat}")

## Các hàm của Lớp
Hãy cùng kiểm tra chức năng của các đơn vị (units) để xác định vai trò của chúng trong quyết định chất lượng rang cà phê. Chúng ta sẽ vẽ biểu đồ đầu ra của từng node đối với mọi giá trị của các đặc trưng đầu vào (thời lượng, nhiệt độ). Mỗi đơn vị là một hàm logistic có đầu ra dao động từ 0 đến 1. Vùng bóng mờ trong biểu đồ biểu diễn giá trị đầu ra.
> Lưu ý: Trong các bài thực hành, chúng ta thường đánh số bắt đầu từ 0.

In [ ]:
plt_layer(X,Y.reshape(-1,),W1,b1,norm_l)

Vùng bóng mờ cho thấy mỗi đơn vị chịu trách nhiệm cho một vùng "rang hỏng" (bad roast) khác nhau. Đơn vị 0 có giá trị lớn hơn khi nhiệt độ quá thấp. Đơn vị 1 có giá trị lớn hơn khi thời lượng quá ngắn và đơn vị 2 có giá trị lớn hơn đối với các kết hợp thời gian/nhiệt độ không tốt. Điều đáng chú ý là mạng đã tự học được các hàm này thông qua quá trình hạ gradient. Chúng rất giống với loại hàm mà con người có thể lựa chọn để đưa ra các quyết định tương tự.

Biểu đồ hàm số của lớp cuối cùng hơi khó hình dung hơn một chút. Đầu vào của nó là đầu ra của lớp thứ nhất. Chúng ta biết rằng lớp thứ nhất sử dụng hàm sigmoid nên phạm vi đầu ra của chúng nằm trong khoảng từ 0 đến 1. Chúng ta có thể tạo biểu đồ 3D tính toán đầu ra cho mọi tổ hợp có thể có của ba đầu vào. Điều này được thể hiện bên dưới. Ở trên, các giá trị đầu ra cao tương ứng với các vùng 'rang hỏng'. Ở bên dưới, đầu ra cực đại nằm ở các vùng mà cả ba đầu vào đều có giá trị nhỏ, tương ứng với các vùng 'rang đạt chuẩn'.

In [ ]:
plt_output_unit(W2,b2)

Biểu đồ cuối cùng thể hiện toàn bộ mạng đang hoạt động.  
Biểu đồ bên trái là đầu ra thô của lớp cuối cùng, được biểu diễn bằng vùng màu xanh lam. Biểu đồ này được phủ lên trên dữ liệu huấn luyện (được biểu thị bằng các dấu X và O).   
Biểu đồ bên phải là đầu ra của mạng sau khi áp dụng ngưỡng quyết định. Các dấu X và O ở đây tương ứng với các quyết định do mạng đưa ra.  
Đoạn code sau đây sẽ mất một chút thời gian để chạy:

In [ ]:
def netf(x):
    model.eval()
    with torch.no_grad():
        x_norm = torch.tensor(norm_l(x), dtype=torch.float32)
        return model(x_norm).numpy()

plt_network(X,Y,netf)

## Chúc mừng bạn!
Bạn đã xây dựng thành công một mạng neural đơn giản bằng **PyTorch**.

*Nguồn tham khảo: Andrew Ng, Advanced Learning Algorithms, DeepLearning.AI & Standford Online.*